# SOTA Baseline: YOLOFM (2024) on FLAME Dataset

Purpose: Train YOLOFM (2024) adapted for FLAME segmentation as a DICTA 2026 baseline.

Key Features:
- FocalNext Backbone with focal-level depth-wise convolution
- QAHARep-FPN Neck with hierarchical attention
- Lightweight segmentation head
- Focal-SIoU Loss (combines focal weighting + spatial IoU)
- SGD optimizer (lr=0.01, momentum=0.937)
- Max 80 epochs with early stopping (patience=15)

## 1. Setup: Imports and Paths

In [1]:
import json
import time
import warnings
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
import sys
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts' / 'data'))
sys.path.insert(0, str(PROJECT_ROOT / 'models' / 'sota_baselines'))

from flame_dataset import FLAMEDataset
from yolofm_2024 import create_yolofm_2024, FocalSIoULoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device: {device}')
print(f'Project root: {PROJECT_ROOT}')

Device: cpu
Project root: c:\SPJAIN\BushFire-Detection


## 2. Configuration

In [2]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'Output' / 'Segmentation_Augmented'
CKPT_DIR = PROJECT_ROOT / 'models' / 'trained' / 'sota_baselines' / 'yolofm_2024'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT_PATH = CKPT_DIR / 'best_model.pth'
METRICS_OUT = PROJECT_ROOT / 'data' / 'processed' / 'Output' / 'sota_yolofm_metrics.json'

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 80
NUM_WORKERS = 0
LR = 0.01
MOMENTUM = 0.937
WEIGHT_DECAY = 5e-4
PATIENCE = 15

print('YOLOFM (2024) Training Configuration')
print(f'DATA_DIR exists: {DATA_DIR.exists()}')
print(f'Checkpoint path: {BEST_CKPT_PATH}')
print(f'Optimizer: SGD lr={LR} momentum={MOMENTUM}')
print(f'Max epochs: {EPOCHS} | Patience: {PATIENCE}')

YOLOFM (2024) Training Configuration
DATA_DIR exists: True
Checkpoint path: c:\SPJAIN\BushFire-Detection\models\trained\sota_baselines\yolofm_2024\best_model.pth
Optimizer: SGD lr=0.01 momentum=0.937
Max epochs: 80 | Patience: 15


## 3. Dataset and DataLoaders

In [3]:
train_dataset = FLAMEDataset(
    root_dir=DATA_DIR,
    split='train',
    img_size=IMG_SIZE,
    train_ratio=0.8,
    augment=True,
)

val_dataset = FLAMEDataset(
    root_dir=DATA_DIR,
    split='val',
    img_size=IMG_SIZE,
    train_ratio=0.8,
    augment=False,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'), drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda')
)

sample = next(iter(train_loader))
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Image tensor shape: {sample["image"].shape}')

FLAME Dataset (train): 646 samples, size=256x256, augment=True
FLAME Dataset (val): 162 samples, size=256x256, augment=False
Train samples: 646 | Val samples: 162
Train batches: 40 | Val batches: 11
Image tensor shape: torch.Size([16, 3, 256, 256])


## 4. Build YOLOFM (2024)

In [4]:
model = create_yolofm_2024(in_channels=3, out_channels=1).to(device)
stats = model.get_parameter_count()

print(f'YOLOFM (2024)')
print(f'Total params: {stats["total_parameters"]:,}')
print(f'Model size: {stats["model_size_mb"]:.2f} MB')

# Test forward pass
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    y = model(x)
print(f'Forward output shape: {y.shape}')
print(f'Output range: [{y.min():.3f}, {y.max():.3f}]')

YOLOFM (2024)
Total params: 4,681,945
Model size: 17.86 MB
Forward output shape: torch.Size([2, 1, 256, 256])
Output range: [0.229, 0.823]


## 5. Loss, Metrics, Optimizer

In [5]:
# Focal-SIoU Loss
loss_fn = FocalSIoULoss(alpha=0.5, gamma=2.0)

# Metrics
def compute_metrics(pred: torch.Tensor, target: torch.Tensor, thr: float = 0.5) -> Dict[str, float]:
    pb = (pred > thr).float()
    tb = target.float()
    tp = (pb * tb).sum(dim=[2, 3])
    fp = (pb * (1 - tb)).sum(dim=[2, 3])
    fn = ((1 - pb) * tb).sum(dim=[2, 3])
    iou = tp / (tp + fp + fn + 1e-6)
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    return {'mIoU': float(iou.mean().item()), 'F1': float(f1.mean().item())}

# SGD Optimizer with momentum
optimizer = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

# LR scheduler: linear warmup + cosine annealing
def get_lr_schedule(epoch: int, total_epochs: int, base_lr: float) -> float:
    warmup_epochs = 3
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return base_lr * 0.5 * (1 + np.cos(np.pi * progress))

print('Focal-SIoU Loss + SGD Optimizer initialized')

Focal-SIoU Loss + SGD Optimizer initialized


## 6. Training Loop (80 epochs with Early Stopping)

In [ ]:
best_miou = 0.0
best_epoch = 0
best_f1 = 0.0
history = {'train_loss': [], 'val_miou': [], 'val_f1': []}
patience_counter = 0

print(f'Starting training for {EPOCHS} epochs...')
print('=' * 70)

for epoch in range(EPOCHS):
    # Update LR
    current_lr = get_lr_schedule(epoch, EPOCHS, LR)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    # Training
    model.train()
    train_loss_sum = 0.0

    for batch in train_loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        optimizer.zero_grad()
        pred = model(images)
        loss = loss_fn(pred, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_sum += float(loss.item())

    avg_train_loss = train_loss_sum / max(len(train_loader), 1)

    # Validation
    model.eval()
    val_ious = []
    val_f1s = []
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            pred = model(images)
            m = compute_metrics(pred, masks)
            val_ious.append(m['mIoU'])
            val_f1s.append(m['F1'])

    avg_miou = float(np.mean(val_ious)) if val_ious else 0.0
    avg_f1 = float(np.mean(val_f1s)) if val_f1s else 0.0

    history['train_loss'].append(avg_train_loss)
    history['val_miou'].append(avg_miou)
    history['val_f1'].append(avg_f1)

    # Checkpoint best model
    if avg_miou > best_miou:
        best_miou = avg_miou
        best_f1 = avg_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_miou': best_miou,
            'best_f1': best_f1,
        }, BEST_CKPT_PATH)
    else:
        patience_counter += 1

    # Log
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch + 1:02d}/{EPOCHS} | LR={current_lr:.5f} | '
              f'train_loss={avg_train_loss:.4f} | val_mIoU={avg_miou:.4f} | val_F1={avg_f1:.4f}')

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch + 1} (no improvement for {PATIENCE} epochs)')
        break

print('=' * 70)
print(f'Training completed. Best model at epoch {best_epoch + 1}')
print(f'Best mIoU: {best_miou:.4f} | Best F1: {best_f1:.4f}')
print(f'Checkpoint saved: {BEST_CKPT_PATH}')

Starting training for 80 epochs...


## 7. Load Best Model and Compute Benchmarking Metrics

In [ ]:
checkpoint = torch.load(BEST_CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
model.eval()

print(f'Loaded best model from epoch {checkpoint["epoch"] + 1}')
print(f'Best validation mIoU: {checkpoint["best_miou"]:.4f}')

## 8. FLOPs and Latency Benchmark

In [ ]:
def estimate_flops(model: nn.Module, input_shape: Tuple) -> float:
    hooks = []
    total_ops = {'v': 0}

    def hook(m, inp, out):
        if isinstance(m, nn.Conv2d):
            out_h, out_w = out.shape[-2], out.shape[-1]
            kernel_ops = m.kernel_size[0] * m.kernel_size[1] * (m.in_channels / m.groups)
            ops = out_h * out_w * m.out_channels * kernel_ops
            total_ops['v'] += ops

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(hook))

    x = torch.randn(*input_shape).to(device)
    with torch.no_grad():
        _ = model(x)

    for h in hooks:
        h.remove()

    return float(total_ops['v'] / 1e9)

def benchmark_latency(model: nn.Module, input_shape=(1, 3, 256, 256), warmup=20, runs=100) -> Dict[str, float]:
    x = torch.randn(*input_shape).to(device)
    times = []

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)

        for _ in range(runs):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000.0)

    arr = np.array(times)
    return {
        'mean_ms': float(arr.mean()),
        'p95_ms': float(np.percentile(arr, 95)),
        'p99_ms': float(np.percentile(arr, 99)),
    }

params_m = stats['total_parameters'] / 1e6
flops_g = estimate_flops(model, (1, 3, IMG_SIZE, IMG_SIZE))
lat = benchmark_latency(model, (1, 3, IMG_SIZE, IMG_SIZE), warmup=20, runs=100)

print(f'Params (M): {params_m:.3f}')
print(f'FLOPs (G): {flops_g:.3f}')
print(f'Latency P95 (ms): {lat["p95_ms"]:.2f}')
print(f'Latency P99 (ms): {lat["p99_ms"]:.2f}')

## 9. Final Validation and Export Metrics

In [ ]:
model.eval()
final_ious = []
final_f1s = []

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)
        pred = model(images)
        m = compute_metrics(pred, masks)
        final_ious.append(m['mIoU'])
        final_f1s.append(m['F1'])

final_miou = float(np.mean(final_ious)) if final_ious else 0.0
final_f1 = float(np.mean(final_f1s)) if final_f1s else 0.0

results = {
    'model_name': 'YOLOFM (2024)',
    'dataset': 'FLAME Augmented',
    'checkpoint_path': str(BEST_CKPT_PATH),
    'metrics': {
        'miou': final_miou,
        'f1': final_f1,
        'params_m': float(params_m),
        'flops_g': float(flops_g),
        'latency_mean_ms': float(lat['mean_ms']),
        'latency_p95_ms': float(lat['p95_ms']),
        'latency_p99_ms': float(lat['p99_ms']),
    },
    'training': {
        'optimizer': 'SGD',
        'learning_rate': LR,
        'momentum': MOMENTUM,
        'loss': 'Focal-SIoU',
        'epochs_planned': EPOCHS,
        'epochs_run': len(history['train_loss']),
        'patience': PATIENCE,
    },
    'dataset_split': {
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
        'total_samples': len(train_dataset) + len(val_dataset),
    }
}

METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(METRICS_OUT, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Metrics exported to: {METRICS_OUT}')
print(json.dumps(results, indent=2))

## 10. DICTA Benchmark Table Row

In [ ]:
import pandas as pd

row = {
    'Model': 'YOLOFM (2024)',
    'Type': 'SOTA Baseline',
    'Dataset': 'FLAME',
    'Params (M)': f'{params_m:.3f}',
    'FLOPs (G)': f'{flops_g:.3f}',
    'Size (MB)': f'{stats["model_size_mb"]:.2f}',
    'mIoU': f'{final_miou:.4f}',
    'F1': f'{final_f1:.4f}',
    'P95 Latency (ms)': f'{lat["p95_ms"]:.2f}',
}

df = pd.DataFrame([row])
print('\n' + '=' * 120)
print('DICTA BENCHMARK TABLE ROW')
print('=' * 120)
print(df.to_string(index=False))
print('=' * 120)